In [65]:
# importowanie bibliotek

import numpy as np
import pandas as pd

import os

from pathlib import Path


from Bio.PDB import PDBParser




In [51]:
# nazwy dir
main_dir=Path(f"/mnt/storage_10T/Sano_internship/MariaWiluszData")

data_dir=Path(f"{main_dir}/data")

results_dir=Path(f"{main_dir}/all_duf_results")



# Sprawdzanie plddt

Zwracanie csv ze średnim plddt dla całych białek i domen z Pfam i Chainsaw, żeby dało się sprawdzić ile jest takich ze średnim plddt poniżej 70. Wtedy można je też odfiltrować albo zrobić na nich DeepFRI na samej sekwencji a nie strukturze

In [52]:
# list wystarczy bo potem i tak się przeszukuje cały folder

def domains_to_list(file_name):
    '''Funkcja zamieniająca csv z kolumną domain_number w listę numerów domen'''
    
    return list(set(pd.read_csv(file_name)["domain_number"]))
    

In [80]:
def return_plddts(domain_list,dir_type,is_known):
    ''' returns a csv with all subaccessions (later called accession) for each domain in domain_list and their mean_plddt (for each subaccession),
    dir_type must be one of: ["chainsaw","pfam","full"],
    is_known must be "known" or "unknown"
    columns in the saved csv are: ["domain_number", "accession", "mean_plddt"]'''
    
    parser=PDBParser(QUIET=True)
    results=[]
    for domain in domain_list:
        if dir_type=="chainsaw":
            accessions_dir=Path(f"{data_dir}/{domain}_chainsaw_structures")
        elif dir_type=="pfam":
            accessions_dir=Path(f"{data_dir}/{domain}_extracted_structures")
        elif dir_type=="full":
            accessions_dir=Path(f"{data_dir}/{domain}_full_structures")
        else:
            raise ValueError("Error in type of directory, dir_type must be one of: ['chainsaw','pfam','full']")
        try:
            
            
            for file in os.listdir(accessions_dir):
                if file.endswith(".pdb"):
                    if dir_type=="full":
                        accession=file.split(".pdb")[0]
                    else:
                        #rsplit gdyby w accession było "_fragment_" bo rsplit działa od tyłu, a ",1" żeby zadziałał tylko raz
                        accession=file.rsplit("_fragment_",1)[0]



                    structure=parser.get_structure(file,os.path.join(accessions_dir,file))
                    plddts=[]
                    for residue in structure.get_residues():
                        atoms=list(residue.get_atoms())
                        if atoms:
                            plddt=atoms[0].get_bfactor() # bo kolumna b-factor z plików pdb z AlphaFold ma informacje o plddt
                            plddts.append(plddt)
                    if plddts:
                        mean_plddt=round(sum(plddts)/len(plddts),2)
                        results.append([domain,accession,mean_plddt])
                
                    
        except:
            print(f"Error for {domain}")
            
    df=pd.DataFrame(results,columns=["domain_number","accession","mean_plddt"])
    df.to_csv(f"{results_dir}/{is_known}_{dir_type}_mean_plddts",index=False)
                    

In [83]:
list_known=domains_to_list(f"{main_dir}/accessions_random_100_known_function.csv")
list_unknown=domains_to_list(f"{main_dir}/accessions_random_100_unknown_function.csv")



## Całe białka

In [82]:
# domeny o znanej funkcji

return_plddts(list_known,dir_type="full",is_known="known")



In [97]:
df=pd.read_csv(f"{results_dir}/known_full_mean_plddts")

df_good_structures=df[df["mean_plddt"]>70]

print(f"Percent of proteins with mean plddt>70: {round(100*len(df_good_structures)/len(df),2)}")


Percent of proteins with mean plddt>70: 85.99


In [96]:
# dufy

return_plddts(list_unknown,dir_type="full",is_known="unknown")



In [98]:
df=pd.read_csv(f"{results_dir}/unknown_full_mean_plddts")

df_good_structures=df[df["mean_plddt"]>70]

print(f"Percent of proteins with mean plddt>70: {round(100*len(df_good_structures)/len(df),2)}")


Percent of proteins with mean plddt>70: 86.62


## Domeny z Pfam

In [87]:
# domeny o znanej funkcji

return_plddts(list_known,dir_type="pfam",is_known="known")



In [99]:
df=pd.read_csv(f"{results_dir}/known_pfam_mean_plddts")

df_good_structures=df[df["mean_plddt"]>70]

print(f"Percent of domains with mean plddt>70: {round(100*len(df_good_structures)/len(df),2)}")


Percent of domains with mean plddt>70: 94.38


In [88]:
# dufy

return_plddts(list_unknown,dir_type="pfam",is_known="unknown")



In [100]:
df=pd.read_csv(f"{results_dir}/unknown_pfam_mean_plddts")

df_good_structures=df[df["mean_plddt"]>70]

print(f"Percent of domains with mean plddt>70: {round(100*len(df_good_structures)/len(df),2)}")


Percent of domains with mean plddt>70: 92.19


## Domeny z Chainsaw

In [89]:
# domeny o znanej funkcji

return_plddts(list_known,dir_type="chainsaw",is_known="known")



In [101]:
df=pd.read_csv(f"{results_dir}/known_chainsaw_mean_plddts")

df_good_structures=df[df["mean_plddt"]>70]

print(f"Percent of domains with mean plddt>70: {round(100*len(df_good_structures)/len(df),2)}")


Percent of domains with mean plddt>70: 93.39


In [90]:
# dufy

return_plddts(list_unknown,dir_type="chainsaw",is_known="unknown")



In [102]:
df=pd.read_csv(f"{results_dir}/unknown_chainsaw_mean_plddts")

df_good_structures=df[df["mean_plddt"]>70]

print(f"Percent of domains with mean plddt>70: {round(100*len(df_good_structures)/len(df),2)}")


Percent of domains with mean plddt>70: 95.33


# Podsumowanie

Conajmniej 85% białek i 92% domen ma dobre struktury, więc narazie można tylko poodfiltrowywać najwyżej te pozostałe.
